In [ ]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
# 
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    f1_score,
    classification_report
)

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce MX450


In [7]:
train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/validation.csv")
test_df = pd.read_csv("../data/processed/test.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (4135, 3)
Validation: (517, 3)
Test: (517, 3)


In [8]:
train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/validation.csv")
test_df = pd.read_csv("../data/processed/test.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (4135, 3)
Validation: (517, 3)
Test: (517, 3)


In [9]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={
        0: "ham",
        1: "spam"
    },
    label2id={
        "ham": 0,
        "spam": 1
    }
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
MAX_LENGTH = 128

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

In [ ]:
train_dataset = Dataset.from_pandas(
    train_df[["text", "label_id"]],
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df[["text", "label_id"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["text", "label_id"]],
    preserve_index=False
)

In [12]:
train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True
)

val_tokenized = val_dataset.map(
    tokenize_function,
    batched=True
)

test_tokenized = test_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/4135 [00:00<?, ? examples/s]

Map:   0%|          | 0/517 [00:00<?, ? examples/s]

Map:   0%|          | 0/517 [00:00<?, ? examples/s]

In [13]:
train_tokenized = train_tokenized.rename_column(
    "label_id",
    "labels"
)

val_tokenized = val_tokenized.rename_column(
    "label_id",
    "labels"
)

test_tokenized = test_tokenized.rename_column(
    "label_id",
    "labels"
)

In [14]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=4,
    lora_alpha=8,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
    bias="none"
)

In [15]:
model = get_peft_model(
    model,
    lora_config
)

In [16]:
model.print_trainable_parameters()

trainable params: 665,858 || all params: 67,620,868 || trainable%: 0.9847


In [17]:
trainable_names = [
    name
    for name, param in model.named_parameters()
    if param.requires_grad
]

trainable_names[:30]

['base_model.model.distilbert.transformer.layer.0.attention.q_lin.lora_A.default.weight',
 'base_model.model.distilbert.transformer.layer.0.attention.q_lin.lora_B.default.weight',
 'base_model.model.distilbert.transformer.layer.0.attention.v_lin.lora_A.default.weight',
 'base_model.model.distilbert.transformer.layer.0.attention.v_lin.lora_B.default.weight',
 'base_model.model.distilbert.transformer.layer.1.attention.q_lin.lora_A.default.weight',
 'base_model.model.distilbert.transformer.layer.1.attention.q_lin.lora_B.default.weight',
 'base_model.model.distilbert.transformer.layer.1.attention.v_lin.lora_A.default.weight',
 'base_model.model.distilbert.transformer.layer.1.attention.v_lin.lora_B.default.weight',
 'base_model.model.distilbert.transformer.layer.2.attention.q_lin.lora_A.default.weight',
 'base_model.model.distilbert.transformer.layer.2.attention.q_lin.lora_B.default.weight',
 'base_model.model.distilbert.transformer.layer.2.attention.v_lin.lora_A.default.weight',
 'base_mod

In [18]:
model.print_trainable_parameters()

trainable params: 665,858 || all params: 67,620,868 || trainable%: 0.9847


In [19]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        zero_division=0
    )

    macro_f1 = f1_score(
        labels,
        predictions,
        average="macro"
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "spam_f1": f1,
        "macro_f1": macro_f1
    }

In [20]:
training_args = TrainingArguments(
    output_dir="../results/lora_r4",

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    learning_rate=2e-5,
    weight_decay=0.01,

    num_train_epochs=3,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    logging_steps=25,

    seed=42,

    report_to="none",

    fp16=True
)

In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [22]:
trainer.train()

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,Spam F1,Macro F1
1,0.184763,0.151997,0.951644,0.955556,0.651515,0.774775,0.873845
2,0.075055,0.071733,0.978723,0.950820,0.878788,0.913386,0.950629
3,0.049035,0.064299,0.978723,0.936508,0.893939,0.914729,0.951287


TrainOutput(global_step=777, training_loss=0.15326358117594682, metrics={'train_runtime': 590.288, 'train_samples_per_second': 21.015, 'train_steps_per_second': 1.316, 'total_flos': 197885311243776.0, 'train_loss': 0.15326358117594682, 'epoch': 3.0})

In [23]:
test_results = trainer.evaluate(
    test_tokenized
)

test_results

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,Spam F1,Macro F1
0.049035,0.048210,3,0.988395,1.000000,0.907692,0.951613,0.972510


{'eval_loss': 0.04820955544710159,
 'eval_accuracy': 0.988394584139265,
 'eval_precision': 1.0,
 'eval_recall': 0.9076923076923077,
 'eval_spam_f1': 0.9516129032258065,
 'eval_macro_f1': 0.9725097483162}

In [24]:
predictions = trainer.predict(test_tokenized)

test_pred = np.argmax(
    predictions.predictions,
    axis=-1
)
test_labels = predictions.label_ids


print(
    classification_report(
        test_labels,
        test_pred,
        target_names=["ham", "spam"]
    )
)

              precision    recall  f1-score   support

         ham       0.99      1.00      0.99       452
        spam       1.00      0.91      0.95        65

    accuracy                           0.99       517
   macro avg       0.99      0.95      0.97       517
weighted avg       0.99      0.99      0.99       517



In [26]:
import time
from pathlib import Path

def run_lora_experiment(r):
    
    print("=" * 60)
    print(f"Starting LoRA experiment: r={r}")
    print("=" * 60)

    # ------------------------------------------------
    # 1. Load a fresh pretrained DistilBERT
    # ------------------------------------------------
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label={
            0: "ham",
            1: "spam"
        },
        label2id={
            "ham": 0,
            "spam": 1
        }
    )

    # ------------------------------------------------
    # 2. Configure LoRA
    # ------------------------------------------------
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=r,
        lora_alpha=2 * r,
        lora_dropout=0.1,
        target_modules=["q_lin", "v_lin"],
        bias="none"
    )

    # ------------------------------------------------
    # 3. Inject LoRA
    # ------------------------------------------------
    model = get_peft_model(
        model,
        lora_config
    )

    # ------------------------------------------------
    # 4. Get parameter counts
    # ------------------------------------------------
    trainable_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    total_params = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable_percent = (
        trainable_params / total_params
    ) * 100

    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable percentage: {trainable_percent:.4f}%")

    # ------------------------------------------------
    # 5. Training arguments
    # ------------------------------------------------
    output_dir = f"../results/lora_r{r}"

    training_args = TrainingArguments(
        output_dir=output_dir,

        per_device_train_batch_size=4,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=4,

        learning_rate=2e-5,
        weight_decay=0.01,

        num_train_epochs=3,

        eval_strategy="epoch",
        save_strategy="epoch",

        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,

        logging_steps=25,

        seed=42,

        report_to="none",

        fp16=True
    )

    # ------------------------------------------------
    # 6. Create Trainer
    # ------------------------------------------------
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=val_tokenized,
        processing_class=tokenizer,
        compute_metrics=compute_metrics
    )

    # ------------------------------------------------
    # 7. Train
    # ------------------------------------------------
    start_time = time.time()

    trainer.train()

    training_time = time.time() - start_time

    # ------------------------------------------------
    # 8. Evaluate on test set
    # ------------------------------------------------
    test_results = trainer.evaluate(
        test_tokenized
    )

    # ------------------------------------------------
    # 9. Save model
    # ------------------------------------------------
    model_dir = Path(
        f"../results/models/lora-r{r}"
    )

    model_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    trainer.save_model(model_dir)
    tokenizer.save_pretrained(model_dir)

    # ------------------------------------------------
    # 10. Collect results
    # ------------------------------------------------
    results = {
        "model": f"LoRA r={r}",
        "rank": r,
        "trainable_params": trainable_params,
        "total_params": total_params,
        "trainable_percent": trainable_percent,
        "accuracy": test_results["eval_accuracy"],
        "precision": test_results["eval_precision"],
        "recall": test_results["eval_recall"],
        "spam_f1": test_results["eval_spam_f1"],
        "macro_f1": test_results["eval_macro_f1"],
        "training_time_sec": training_time
    }

    print("\nTest results:")
    print(results)
    del trainer
    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results

In [32]:
ranks = [4, 8, 16, 32]

all_results = []

for r in ranks:

    result = run_lora_experiment(r)

    all_results.append(result)

    pd.DataFrame(all_results).to_csv(
        "metrics/lora_sweep.csv",
        index=False
    )

Starting LoRA experiment: r=4


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable parameters: 665,858
Total parameters: 67,620,868
Trainable percentage: 0.9847%


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,Spam F1,Macro F1
1,0.661936,0.136164,0.967118,0.962264,0.772727,0.857143,0.919282
2,0.221078,0.064950,0.978723,0.936508,0.893939,0.914729,0.951287
3,0.131480,0.061178,0.976789,0.921875,0.893939,0.907692,0.947209


...

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable parameters: 1,181,954
Total parameters: 68,136,964
Trainable percentage: 1.7347%


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,Spam F1,Macro F1
1,0.238765,0.061259,0.982592,0.938462,0.924242,0.931298,0.960665
2,0.217278,0.062706,0.984526,0.953125,0.924242,0.938462,0.964806
3,0.102335,0.062730,0.984526,0.953125,0.924242,0.938462,0.964806


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,Spam F1,Macro F1
0.102335,0.033499,3,0.994197,1.000000,0.953846,0.976378,0.986535



Test results:
{'model': 'LoRA r=32', 'rank': 32, 'trainable_params': 1181954, 'total_params': 68136964, 'trainable_percent': 1.734673708091837, 'accuracy': 0.9941972920696325, 'precision': 1.0, 'recall': 0.9538461538461539, 'spam_f1': 0.9763779527559056, 'macro_f1': 0.986535172629331, 'training_time_sec': 593.7497339248657}


In [34]:
df = pd.read_csv("metrics/lora_sweep.csv")
df

,model,rank,trainable_params,total_params,trainable_percent,accuracy,precision,recall,spam_f1,macro_f1,training_time_sec
0,LoRA r=4,4,665858,67620868,0.984693,0.986460,0.983333,0.907692,0.944000,0.968150,561.693284
1,LoRA r=8,8,739586,67694596,1.092533,0.990329,0.983871,0.938462,0.960630,0.977559,597.187898
2,LoRA r=16,16,887042,67842052,1.307511,0.992263,1.000000,0.938462,0.968254,0.981924,588.370034
3,LoRA r=32,32,1181954,68136964,1.734674,0.994197,1.000000,0.953846,0.976378,0.986535,593.749734
